# Vector DB

In this notebook, we use a containerized version of Chroma DB. To set up, you will need the following:

1. Install [Docker Desktop](https://www.docker.com/products/docker-desktop/) by following the link and Download Docker Desktop for your operating system.
2. In a terminal window, navigate to the folder ./05_src/chromadb/. For example, on Windows, you would use `cd .\05_src\chromadb`.
3. Run the command `docker compose up -d`, which will start the Chroma DB server.

## Downloading Batch Results

In the previous notebook, we had created batch processes. We will start by consulting the status of our batch processes by identifying them throught their descriptions.

In [ ]:
%load_ext dotenv

import sys
sys.path.append('../../05_src/')
%dotenv ../../05_src/.secrets
%dotenv ../../05_src/.env

In [ ]:
from utils.clients import get_client
from IPython.display import display, Markdown
from tqdm import tqdm
import json
import os

client = get_client(use_gateway=False)
MODEL = os.getenv('MODEL')
EMBEDDING_MODEL = os.getenv('EMBEDDING_MODEL')
DOC_DIR = "../../05_src/documents/pitchfork/"

In [ ]:
batch_description = 'Pitchfork reviews content embeddings (deploying-ai-8192e796b86242b084df7200276ef5f2) 2026-06-13 23:11:19'
batch_processes = client.batches.list().to_dict()
batch_info= [
    {'batch_id': batch['id'],
     'description': batch['metadata']['description'],
    'status': batch['status'],
    'request_counts': batch['request_counts'],
    'output_file_id': batch['output_file_id'],
    'input_file_id': batch['input_file_id']}
            for batch in batch_processes['data'] if batch['metadata']['description'] == batch_description
    ]
batch_info

When the status of the batches is complete, we can query the `output_file_id` where their results will be stored.

More generally, we will require the original text and the embeddings of that original text mapped through the `custom_id`.

In [ ]:
batch_complete = [
    batch  for batch in batch_info if batch['status'] == 'completed'
]
batch_incomplete = [
    batch  for batch in batch_info if batch['status'] != 'completed'
]
print(f"Batch complete: {len(batch_complete)}\n\nBatch incomplete: {len(batch_incomplete)}")

Before we download all results, examine the response of the file API:

In [ ]:

response = client.files.content(batch_complete[0]['output_file_id'])
text_response = response.text
lines = text_response.split('\n')


In [ ]:
json.loads(lines[0])

For our results database, we will need to map the original text to their embeddings. 

In [ ]:
def get_text_and_embeddings(batch):
    embedding_lines =  get_content_from_file(batch, 'output_file_id')
    text_lines = get_content_from_file(batch, 'input_file_id')
    return embedding_lines, text_lines

def get_content_from_file(batch, key):
    file = client.files.content(batch[key])
    text = file.text
    lines = text.split('\n')
    content_lines = [json.loads(line) for line in lines if line.strip()]
    return content_lines


Notice that the response is also a .jsonl file. Therefore, we can process it line-by-line and use the `custom_id` to map to the original document chunk.

The function below:

- Creates a dictionary, `text_dict`, with keys given by each `custom_id` and value equal to the text.
- Iterate over all embedding items and use the dictionary defined above to map the embeddings to their input text.

In [ ]:
def create_chroma_inputs(embedding_lines, text_lines, metadata_dict):
    chroma_inputs = []
    text_dict = {item['custom_id']: item['body']['input'] for item in text_lines}
    for embed_item in embedding_lines:
        custom_id = embed_item['custom_id']
        text = text_dict.get(custom_id, "")
        reviewid = get_reviewid_from_custom_id(custom_id)
        chroma_input = {
            'id': custom_id,
            'embedding': embed_item['response']['body']['data'][0]['embedding'],
            'text': text,
            'metadata': metadata_dict.get(reviewid, {})
        }
        chroma_inputs.append(chroma_input)
    return chroma_inputs

A couple of functions to control the logic flow:

In [ ]:

def process_batch_for_chromadb(batch, metadata_dict):
    embedding_lines, text_lines = get_text_and_embeddings(batch)
    chroma_inputs = create_chroma_inputs(embedding_lines, text_lines, metadata_dict)
    return chroma_inputs

def process_batches_for_chromadb(batches, metadata_dict):
    all_chroma_inputs = []
    for batch in tqdm(batches, desc="Processing batches"):
        chroma_inputs = process_batch_for_chromadb(batch, metadata_dict)
        all_chroma_inputs.extend(chroma_inputs)
    return all_chroma_inputs

def chroma_inputs_to_jsonl(chroma_inputs, output_file):
    with open(output_file, 'w', encoding='utf-8') as f:
        for item in chroma_inputs:
            json_line = json.dumps(item)
            f.write(json_line + '\n')

# Metadata

We enrich each ChromaDB entry with structured metadata joined from the pitchfork source files:

| Field | Source file | Description |
|-------|------------|-------------|
| `reviewid` | pitchfork_reviews | Unique review identifier |
| `album` | pitchfork_reviews | Album title |
| `artist` | pitchfork_reviews | Artist name |
| `score` | pitchfork_reviews | Pitchfork score (0–10) |
| `genre` | pitchfork_genres | Genre(s), comma-separated |
| `label` | pitchfork_labels | Record label |
| `year` | pitchfork_years | Album release year |

Storing metadata directly in ChromaDB allows native filtering with `where` clauses — no additional database query is needed at retrieval time.

In [ ]:
def load_metadata(doc_dir:str) -> dict:
    """Join pitchfork_*.jsonl files into a metadata dict keyed by str(reviewid)."""
    reviews = {str(r['reviewid']): r for r in load_jsonl(os.path.join(doc_dir, 'pitchfork_reviews.jsonl'))}
    genres_raw = load_jsonl(os.path.join(doc_dir, 'pitchfork_genres.jsonl'))
    labels_raw = load_jsonl(os.path.join(doc_dir, 'pitchfork_labels.jsonl'))
    years_raw  = load_jsonl(os.path.join(doc_dir, 'pitchfork_years.jsonl'))

    genres = {}
    for g in genres_raw:
        rid = str(g['reviewid'])
        genres.setdefault(rid, []).append(g.get('genre') or '')

    labels = {}
    for l in labels_raw:
        rid = str(l['reviewid'])
        if rid not in labels:
            labels[rid] = l.get('label') or ''

    years = {str(y['reviewid']): y.get('year') or 0 for y in years_raw}

    metadata = {}
    for rid, r in reviews.items():
        metadata[rid] = {
            'reviewid': rid,
            'album':    str(r.get('title')  or ''),
            'artist':   str(r.get('artist') or ''),
            'score':    float(r.get('score') or 0.0),
            'genre':    ', '.join(g for g in genres.get(rid, []) if g),
            'label':    str(labels.get(rid) or ''),
            'year':     int(years.get(rid) or 0),
        }
    return metadata

def load_jsonl(file:str):
    data = []
    with open(file, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

def get_reviewid_from_custom_id(custom_id:str):
    return custom_id.split('_')[0]

Make sure `chroma_inputs.jsonl` has been placed in `05_src/documents/pitchfork/` (provided by the instructional team). This file contains pre-computed embeddings and metadata for all reviews, so students can load the collection without making any API calls.

Set `use_api = True` to regenerate it from the batch API results instead. Note that batch jobs can take up to 24 hours to complete.

Now, we can create our input dictionaries.

In [ ]:
use_api = False

out_path = os.path.join(DOC_DIR, "chroma_inputs.jsonl")

if use_api:
    os.makedirs(DOC_DIR, exist_ok=True)
    metadata_dict = load_metadata(DOC_DIR)
    chroma_inputs = process_batches_for_chromadb(batch_complete, metadata_dict)
    chroma_inputs_to_jsonl(chroma_inputs, out_path)
else:
    chroma_inputs = []
    with open(out_path, "r", encoding="utf-8") as f:
        for line in f:
            chroma_inputs.append(json.loads(line))

In [ ]:
chroma_inputs[1]

# Load Embeddings to Chroma

In [ ]:
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
import os

COLLECTION_NAME = "pitchfork_reviews"
CHROMA_URL = os.getenv("CHROMA_URL")

def setup_collection(chroma_url:str=CHROMA_URL,
                     collection_name: str = COLLECTION_NAME):
    chroma_client = chromadb.HttpClient(host=chroma_url)
    collections = chroma_client.list_collections()
    if collection_name in [col.name for col in collections]:
        chroma_client.delete_collection(name=collection_name)

    collection = chroma_client.create_collection(
        name=collection_name,
        embedding_function=OpenAIEmbeddingFunction(
            api_key = os.getenv("OPENAI_API_KEY"),
            model_name=EMBEDDING_MODEL)
        )
    return collection

def load_embeddings_to_db(chroma_inputs:list[dict],
                          collection_name:str,
                          chroma_url:str=CHROMA_URL,
                          batch_size:int=1000
                          ):
    collection = setup_collection(chroma_url=chroma_url, collection_name=collection_name)

    for i in tqdm(range(0, len(chroma_inputs), batch_size)):
        batch = chroma_inputs[i:i + batch_size]
        collection.add(
            documents=[item['text'] for item in batch],
            embeddings=[item['embedding'] for item in batch],
            metadatas=[item['metadata'] for item in batch],
            ids=[item['id'] for item in batch]
        )


In [ ]:

load_embeddings_to_db(chroma_inputs=chroma_inputs,
                      collection_name=COLLECTION_NAME,
                      chroma_url=CHROMA_URL, 
                      batch_size=1000)

# Prompt Generator

Here we create a prompt with the context gathered through the different data operations.

In [ ]:
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

USE_GATEWAY = os.getenv("USE_GATEWAY", "False").lower() == "true"

chroma = chromadb.HttpClient(host=CHROMA_URL)
if USE_GATEWAY:
    embedding_function = OpenAIEmbeddingFunction(
        api_key="any value",
        api_base="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
        api_type="openai",
        model_name=EMBEDDING_MODEL,
        default_headers={
            "x-api-key": os.getenv("API_GATEWAY_KEY")
        }
    )
else:
    embedding_function = OpenAIEmbeddingFunction(
        api_key=os.getenv("OPENAI_API_KEY"),
        model_name=EMBEDDING_MODEL
    )

collection = chroma.get_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_function
)

In [ ]:
collection.query(
    query_texts=["A great album with stunning vocals and production."],
    n_results=3
)

In [ ]:
def get_context_data(query:str, collection:chromadb.api.models.Collection, top_n:int):
    results = collection.query(
        query_texts=[query],
        n_results=top_n
    )
    context_data = []
    for idx in range(len(results['ids'][0])):
        details = dict(results['metadatas'][0][idx])
        details['text'] = results['documents'][0][idx]
        context_data.append(details)
    return context_data

def generate_prompt(query:str, collection:chromadb.api.models.Collection, top_n:int):
    context_data = get_context_data(query, collection, top_n)
    prompt = f"Given a query, provide a detailed response using the context from relevant Pitchfork reviews. The context will contain references to {top_n} album reviews.\n\n"
    prompt += f"The score is numeric and its scale is from 0 to 10, with 10 being the highest rating. Any album with a score greater than 8.0 is considered a must-listen; album with a score greater than 6.5 is good.\n\n"
    prompt += f"<query>{query}</query>\n\n"
    prompt += "<context>\n"
    for k, context in enumerate(context_data):
        prompt += f"<album {k}>\n"
        prompt += f"- Album Title: {context.get('album', 'N/A')}\n"
        prompt += f"- Album Artist: {context.get('artist', 'N/A')}\n"
        prompt += f"- Album Score: {context.get('score', 'N/A')}\n"
        prompt += f"- Genre: {context.get('genre', 'N/A')}\n"
        prompt += f"- Label: {context.get('label', 'N/A')}\n"
        prompt += f"- Release Year: {context.get('year', 'N/A')}\n"
        prompt += f"- Review Quote: {context.get('text', 'N/A')}\n"
        prompt += f"</album {k}>\n\n"
    prompt += "</context>\n\n"
    prompt += "\nBased on the context and nothing else, provide a detailed response to the query."
    return prompt

def generate_response(query:str, collection:chromadb.api.models.Collection, top_n:int=1):
    prompt = generate_prompt(query, collection, top_n)
    print("Generated Prompt:\n", prompt)
    response = client.responses.create(
        model=MODEL,
        instructions="You are a helpful assistant that provides information based on Pitchfork reviews.",
        input=[{"role": "user", "content": prompt}],
        max_output_tokens=500,
        temperature=0.7
    )
    return response.output_text


# Query

We can now use chroma's similarity function to query the database. Notice that the query itself needs to be converted to embeddings, so we must provide an `embedding_function`. In this case, we use `OpenAIEmbeddingFunction()` to get compatible embeddings using model `text-embedding-3-small`.

In [ ]:
response = generate_response("What are some highly rated albums by emerging indie artists?", 
                             collection, 
                             3)

In [ ]:
display(Markdown(response))

**Note**: Try changing the top_n parameter to 1 and re-run the query. 